In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import src.imports as lib

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# ĐỌC DỮ LIỆU ĐÃ GHÉP
# ==========================================

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import src.imports as lib

file_path = lib.Path("../data/data-clean/hoc_sinh_lop6_9_clean.xlsx")

df_merged = lib.pd.read_excel(
    file_path,
    engine="openpyxl"
)

print(f"Kích thước dữ liệu: {df_merged.shape}")
display(df_merged.head())

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# KIỂM TRA MISSING VALUES SAU KHI GHÉP
# ==========================================

# Kiểm tra số lượng và tỷ lệ thiếu theo từng cột
missing_report = lib.pd.DataFrame({
    "Số lượng thiếu": df_merged.isna().sum(),
    "Tỷ lệ thiếu (%)": (
        df_merged.isna().mean() * 100
    ).round(2)
})

# Chỉ lấy các cột có giá trị thiếu
missing_cols = missing_report[
    missing_report["Số lượng thiếu"] > 0
].sort_values("Số lượng thiếu", ascending=False)

# Thống kê tổng quan
total_missing = df_merged.isna().sum().sum()
rows_with_missing = df_merged.isna().any(axis=1).sum()
cols_with_missing = df_merged.isna().any(axis=0).sum()

print("=" * 60)
print("KIỂM TRA MISSING VALUES SAU KHI GHÉP")
print("=" * 60)

print(f"Tổng số dòng: {len(df_merged):,}")
print(f"Tổng số cột: {df_merged.shape[1]}")
print(f"Tổng số ô thiếu: {total_missing:,}")
print(f"Số dòng có dữ liệu thiếu: {rows_with_missing:,}")
print(f"Số cột có dữ liệu thiếu: {cols_with_missing:,}")
print(f"Tỷ lệ thiếu toàn bộ: {df_merged.isna().mean().mean() * 100:.2f}%")

if total_missing == 0:
    print("\nPASS: Không phát hiện Missing Values.")
else:
    print("\nFAIL: Phát hiện Missing Values.")
    display(missing_cols)

    # Hiển thị các học sinh có dữ liệu thiếu
    display(
        df_merged.loc[
            df_merged.isna().any(axis=1)
        ]
    )

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# KIỂM TRA DUPLICATE SAU KHI GHÉP
# ==========================================

id_col = "ma_hoc_sinh"

# 1. Kiểm tra dòng trùng hoàn toàn
duplicate_rows = df_merged.duplicated().sum()

# 2. Kiểm tra mã học sinh trùng
duplicate_ids = df_merged[id_col].duplicated().sum()

# 3. Kiểm tra trùng nội dung khi bỏ mã học sinh
duplicate_content = df_merged.drop(
    columns=[id_col]
).duplicated().sum()

# 4. Tổng hợp báo cáo
duplicate_report = lib.pd.DataFrame({
    "Loại kiểm tra": [
        "Dòng trùng hoàn toàn",
        "Mã học sinh trùng",
        "Nội dung trùng (không tính ID)"
    ],
    "Số lượng": [
        duplicate_rows,
        duplicate_ids,
        duplicate_content
    ],
    "Tỷ lệ (%)": [
        duplicate_rows / len(df_merged) * 100,
        duplicate_ids / len(df_merged) * 100,
        duplicate_content / len(df_merged) * 100
    ]
}).round(2)

print("=" * 60)
print("KIỂM TRA DUPLICATE SAU KHI GHÉP")
print("=" * 60)

display(duplicate_report)

# 5. Đánh giá kết quả
if (
    duplicate_rows == 0
    and duplicate_ids == 0
    and duplicate_content == 0
):
    print("PASS: Không phát hiện dữ liệu trùng lặp.")
else:
    print("FAIL: Phát hiện dữ liệu trùng lặp.")

    # Hiển thị các dòng có mã học sinh trùng
    if duplicate_ids > 0:
        print("\nCác học sinh có mã trùng:")
        display(
            df_merged[
                df_merged[id_col].duplicated(keep=False)
            ]
        )

    # Hiển thị các dòng có nội dung trùng (không tính ID)
    if duplicate_content > 0:
        print("\nCác dòng có nội dung trùng:")
        display(
            df_merged[
                df_merged.drop(columns=[id_col])
                .duplicated(keep=False)
            ]
        )

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# KIỂM TRA ĐIỂM NẰM TRONG THANG 0–10
# ==========================================

# 1. Xác định 96 cột điểm
score_cols = [
    col for col in df_merged.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

# 2. Kiểm tra giá trị không phải số và điểm ngoài [0, 10]
scores = df_merged[score_cols].apply(
    lib.pd.to_numeric, errors="coerce"
)

invalid_mask = (
    scores.lt(0)
    | scores.gt(10)
    | (scores.isna() & df_merged[score_cols].notna())
)

# 3. Thống kê theo từng cột
invalid_report = lib.pd.DataFrame({
    "Số giá trị không hợp lệ": invalid_mask.sum(),
    "Tỷ lệ (%)": (
        invalid_mask.mean() * 100
    ).round(2)
})

invalid_report = invalid_report[
    invalid_report["Số giá trị không hợp lệ"] > 0
]

# 4. Tổng hợp kết quả
total_invalid = int(invalid_mask.sum().sum())
students_invalid = int(invalid_mask.any(axis=1).sum())

print("=" * 60)
print("KIỂM TRA ĐIỂM TRONG THANG 0–10")
print("=" * 60)

print(f"Số cột điểm kiểm tra: {len(score_cols)}")
print(f"Tổng giá trị không hợp lệ: {total_invalid:,}")
print(f"Số học sinh có điểm không hợp lệ: {students_invalid:,}")

# 5. Đánh giá
if total_invalid == 0:
    print("\nPASS: Tất cả giá trị điểm không thiếu đều hợp lệ trong thang 0–10.")
else:
    print("\nFAIL: Phát hiện giá trị điểm không hợp lệ.")

    display(invalid_report)

    # Hiển thị chi tiết các giá trị không hợp lệ
    invalid_details = (
        df_merged[score_cols]
        .where(invalid_mask)
        .stack()
        .reset_index()
    )

    invalid_details.columns = [
        "Dòng", "Cột điểm", "Giá trị không hợp lệ"
    ]

    invalid_details.insert(
        0,
        "ma_hoc_sinh",
        df_merged.loc[
            invalid_details["Dòng"], "ma_hoc_sinh"
        ].to_numpy()
    )

    display(invalid_details)

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# KIỂM TRA GIÁ TRỊ NHÃN 0/1
# ==========================================

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

# Kiểm tra giá trị nhãn hợp lệ
invalid_mask = ~df_merged[choice_cols].isin([0, 1])

invalid_report = lib.pd.DataFrame({
    "Số giá trị không hợp lệ": invalid_mask.sum(),
    "Tỷ lệ (%)": (invalid_mask.mean() * 100).round(2)
})

print("=" * 60)
print("KIỂM TRA GIÁ TRỊ NHÃN 0/1")
print("=" * 60)

print(f"Tổng giá trị không hợp lệ: {invalid_mask.sum().sum():,}")
print(f"Số học sinh có nhãn không hợp lệ: {invalid_mask.any(axis=1).sum():,}")

if not invalid_mask.any().any():
    print("\nPASS: Tất cả nhãn chỉ chứa 0 và 1.")
else:
    print("\nFAIL: Phát hiện nhãn không hợp lệ.")
    display(invalid_report[invalid_report["Số giá trị không hợp lệ"] > 0])
    display(df_merged.loc[invalid_mask.any(axis=1), ["ma_hoc_sinh"] + choice_cols])

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# KIỂM TRA RÀNG BUỘC CHỌN ĐÚNG 4/9 MÔN
# ==========================================

labels = df_merged[choice_cols]

# Chỉ chấp nhận 9 nhãn hợp lệ và đúng 4 nhãn bằng 1
valid_binary = labels.isin([0, 1]).all(axis=1)
selected_count = labels.eq(1).sum(axis=1)

valid_selection = valid_binary & selected_count.eq(4)

print("=" * 60)
print("KIỂM TRA RÀNG BUỘC 4/9 MÔN")
print("=" * 60)

print(f"Tổng số học sinh: {len(df_merged):,}")
print(f"Số học sinh hợp lệ: {valid_selection.sum():,}")
print(f"Số học sinh không hợp lệ: {(~valid_selection).sum():,}")
print(f"Tỷ lệ hợp lệ: {valid_selection.mean() * 100:.2f}%")

print("\nPhân phối số môn được chọn:")
display(selected_count.value_counts().sort_index().to_frame("Số học sinh"))

if valid_selection.all():
    print("\nPASS: Tất cả học sinh có đúng 4/9 nhãn bằng 1.")
else:
    print("\nFAIL: Phát hiện học sinh không thỏa điều kiện.")
    display(
        df_merged.loc[
            ~valid_selection,
            ["ma_hoc_sinh"] + choice_cols
        ]
    )

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# KIỂM TRA CÔNG THỨC ĐIỂM CẢ NĂM
# ==========================================

subjects = [
    "van", "toan", "ngoai_ngu_1", "gdcd",
    "lich_su_dia_li", "khtn", "cong_nghe", "tin_hoc"
]

consistency_reports = []
inconsistent_details = []

for grade in [6, 7, 8, 9]:
    for subject in subjects:

        col_hki = f"L{grade}_{subject}_hki"
        col_hkii = f"L{grade}_{subject}_hkii"
        col_year = f"L{grade}_{subject}_ca_nam"

        # Chỉ kiểm tra các dòng có đủ ba điểm
        valid = df_merged[[col_hki, col_hkii, col_year]].notna().all(axis=1)

        # ROUND_HALF_UP cho điểm không âm, tránh sai khác do làm tròn bankers
        expected = lib.np.floor(
            (
                df_merged.loc[valid, col_hki]
                + 2 * df_merged.loc[valid, col_hkii]
            ) / 3 * 10 + 0.5
        ) / 10

        actual = df_merged.loc[valid, col_year]

        mismatch = ~lib.np.isclose(
            actual.to_numpy(),
            expected.to_numpy(),
            atol=1e-9,
            rtol=0
        )

        error_count = int(mismatch.sum())

        consistency_reports.append({
            "Lớp": grade,
            "Môn học": subject,
            "Số dòng kiểm tra": int(valid.sum()),
            "Số dòng sai công thức": error_count,
            "Tỷ lệ sai (%)": round(
                error_count / valid.sum() * 100, 2
            ) if valid.sum() else 0
        })

        if error_count > 0:
            error_index = actual.index[mismatch]

            details = df_merged.loc[
                error_index,
                ["ma_hoc_sinh", col_hki, col_hkii, col_year]
            ].copy()

            details["Điểm cả năm tính lại"] = expected[mismatch]
            details["Lớp"] = grade
            details["Môn học"] = subject

            inconsistent_details.append(details)

# Tổng hợp kết quả
consistency_df = lib.pd.DataFrame(consistency_reports)

print("=" * 60)
print("KIỂM TRA TÍNH NHẤT QUÁN ĐIỂM HỌC KỲ VÀ CẢ NĂM")
print("=" * 60)

print(f"Tổng số phép kiểm tra: {consistency_df['Số dòng kiểm tra'].sum():,}")
print(f"Tổng số trường hợp sai: {consistency_df['Số dòng sai công thức'].sum():,}")

display(consistency_df)

if consistency_df["Số dòng sai công thức"].sum() == 0:
    print("\nPASS: Điểm cả năm nhất quán với HKI và HKII.")
else:
    print("\nFAIL: Phát hiện điểm cả năm không khớp công thức.")
    display(lib.pd.concat(inconsistent_details, ignore_index=True))

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# GHI NHẬN OUTLIERS TỪ EDA
# ==========================================

outlier_report = lib.pd.DataFrame([{
    "Phương pháp": "IQR",
    "Số giá trị ngoại lệ": 3288,
    "Số học sinh có ngoại lệ": 482,
    "Tỷ lệ học sinh (%)": 4.82,
    "Hướng xử lý": "Giữ nguyên dữ liệu",
    "Nguồn": "Kết quả EDA"
}])

display(outlier_report)

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# LƯU BÁO CÁO KIỂM ĐỊNH DỮ LIỆU
# ==========================================

report_dir = lib.Path("../data/report-of-data")
report_dir.mkdir(parents=True, exist_ok=True)

report_file = report_dir / "data_validation_report.xlsx"

# Xác định các cột điểm
score_cols = [
    col for col in df_merged.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

scores = df_merged[score_cols].apply(
    lib.pd.to_numeric, errors="coerce"
)

score_invalid = (
    scores.lt(0)
    | scores.gt(10)
    | (scores.isna() & df_merged[score_cols].notna())
)

labels = df_merged[choice_cols]
valid_binary = labels.isin([0, 1]).all(axis=1)
valid_four = valid_binary & labels.eq(1).sum(axis=1).eq(4)

# Tổng hợp kiểm định
validation_summary = lib.pd.DataFrame([
    {
        "Tiêu chí": "Kích thước dữ liệu",
        "Kết quả": str(df_merged.shape),
        "Trạng thái": "PASS" if df_merged.shape == (10000, 106) else "FAIL"
    },
    {
        "Tiêu chí": "Missing Values",
        "Kết quả": int(df_merged.isna().sum().sum()),
        "Trạng thái": "PASS" if not df_merged.isna().any().any() else "FAIL"
    },
    {
        "Tiêu chí": "Duplicate ID",
        "Kết quả": int(df_merged["ma_hoc_sinh"].duplicated().sum()),
        "Trạng thái": "PASS" if df_merged["ma_hoc_sinh"].is_unique else "FAIL"
    },
    {
        "Tiêu chí": "Duplicate toàn bộ dòng",
        "Kết quả": int(df_merged.duplicated().sum()),
        "Trạng thái": "PASS" if not df_merged.duplicated().any() else "FAIL"
    },
    {
        "Tiêu chí": "Điểm ngoài thang 0–10",
        "Kết quả": int(score_invalid.sum().sum()),
        "Trạng thái": "PASS" if not score_invalid.any().any() else "FAIL"
    },
    {
        "Tiêu chí": "Nhãn không thuộc {0, 1}",
        "Kết quả": int((~labels.isin([0, 1])).sum().sum()),
        "Trạng thái": "PASS" if valid_binary.all() else "FAIL"
    },
    {
        "Tiêu chí": "Học sinh không thỏa 4/9 nhãn",
        "Kết quả": int((~valid_four).sum()),
        "Trạng thái": "PASS" if valid_four.all() else "FAIL"
    },
    {
        "Tiêu chí": "Điểm cả năm sai công thức",
        "Kết quả": int(consistency_df["Số dòng sai công thức"].sum()),
        "Trạng thái": (
            "PASS"
            if consistency_df["Số dòng sai công thức"].sum() == 0
            else "FAIL"
        )
    },
    {
        "Tiêu chí": "Giá trị ngoại lệ từ EDA",
        "Kết quả": 3288,
        "Trạng thái": "GHI NHẬN - GIỮ NGUYÊN"
    }
])

# Lưu báo cáo Excel nhiều sheet
with lib.pd.ExcelWriter(report_file, engine="openpyxl") as writer:

    validation_summary.to_excel(
        writer, sheet_name="Tong_hop", index=False
    )

    missing_report.to_excel(
        writer, sheet_name="Missing_Values"
    )

    duplicate_report.to_excel(
        writer, sheet_name="Duplicate", index=False
    )

    consistency_df.to_excel(
        writer, sheet_name="Diem_ca_nam", index=False
    )

    outlier_report.to_excel(
        writer, sheet_name="Outliers_EDA", index=False
    )

print(f"Đã lưu báo cáo: {report_file.resolve()}")
display(validation_summary)